# 00 — Kaggle Dataset Exploration
**No API calls required.** Uses `data/kaggle/dataset.csv` only.

114K Spotify tracks with full audio features — this is our foundation for Signal 1 (catalog coherence).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

FIGURES = Path('../paper/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

AUDIO_FEATURES = ['danceability','energy','loudness','speechiness',
                  'acousticness','instrumentalness','liveness','valence','tempo']
FEATURE_4D = ['danceability','energy','valence','acousticness']

print('Setup OK')

## 1. Load & Basic Stats

In [ ]:
df = pd.read_csv('../data/kaggle/dataset.csv')
print('Shape:', df.shape)
print()
print('Columns:', df.columns.tolist())
print()
print('Dtypes:')
print(df.dtypes)
print()
print('First 5 rows:')
df.head()

In [ ]:
print('=== describe() — all numeric columns ===')
df.describe()

In [ ]:
print('=== Missing values ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f'\nTotal nulls: {nulls.sum()}')

In [ ]:
print(f'Track ID column:  track_id')
print(f'Artist column:    artists')
print(f'Unique artists:   {df["artists"].nunique():,}')
print(f'Unique track IDs: {df["track_id"].nunique():,}')
print(f'Total rows:       {len(df):,}')
print(f'Unique genres:    {df["track_genre"].nunique()}')
print(f'\nAll genres:')
print(sorted(df['track_genre'].dropna().unique().tolist()))

## 2. Distribution Plots

In [ ]:
# 2a. Histogram of 4 core features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']

for ax, feat, color in zip(axes.flatten(), FEATURE_4D, colors):
    ax.hist(df[feat].dropna(), bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(feat.capitalize(), fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'mean={df[feat].mean():.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Core Audio Features (114K Tracks)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_feature_distributions.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: kaggle_feature_distributions.png')

In [ ]:
# 2b. Correlation heatmap of all audio features
feat_cols = [f for f in AUDIO_FEATURES if f in df.columns]
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})
ax.set_title('Audio Feature Correlation Matrix (114K Tracks)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_correlation_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: kaggle_correlation_heatmap.png')
print('\nStrongest correlations:')
corr_pairs = corr.unstack().drop_duplicates().sort_values(key=abs, ascending=False)
corr_pairs = corr_pairs[(corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1))]
print(corr_pairs.head(10))

In [ ]:
# 2c. Scatter: energy vs danceability, colored by valence
sample = df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(sample['danceability'], sample['energy'],
                c=sample['valence'], cmap='RdYlGn',
                alpha=0.5, s=10, edgecolors='none')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Valence (0=sad, 1=happy)', fontsize=11)
ax.set_xlabel('Danceability', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title('Energy vs Danceability (colored by Valence)\n5K random sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_energy_dance_scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: kaggle_energy_dance_scatter.png')

In [ ]:
# 2d. Box plot of 4 core features
fig, ax = plt.subplots(figsize=(10, 6))
data_to_plot = [df[f].dropna().values for f in FEATURE_4D]
bp = ax.boxplot(data_to_plot, labels=[f.capitalize() for f in FEATURE_4D],
                patch_artist=True, notch=True)
colors_bp = ['#3498db','#e74c3c','#2ecc71','#f39c12']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Feature Value (normalized 0–1)', fontsize=11)
ax.set_title('Distribution of Core Audio Features — Box Plot', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_boxplot.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: kaggle_boxplot.png')

## 3. Artists with Most Tracks

In [ ]:
top20_by_count = df['artists'].value_counts().head(20)
print('Top 20 artists by track count:')
print(top20_by_count.to_string())

fig, ax = plt.subplots(figsize=(10, 7))
top20_by_count.sort_values().plot(kind='barh', ax=ax, color='#3498db', alpha=0.8)
ax.set_xlabel('Track Count in Kaggle Dataset')
ax.set_title('Top 20 Artists by Track Count', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_top20_artists.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Low-Variance Artists — Potential Ghost Candidates

In [ ]:
# Compute per-artist variance for artists with 10+ tracks
artist_groups = df.groupby('artists')
artist_counts = artist_groups.size()
artists_10plus = artist_counts[artist_counts >= 10].index

print(f'Artists with 10+ tracks: {len(artists_10plus):,}')

rows = []
for artist in artists_10plus:
    grp = df[df['artists'] == artist]
    row = {'artist': artist, 'track_count': len(grp)}
    for f in FEATURE_4D:
        row[f'var_{f}'] = float(grp[f].var())
    row['total_variance'] = sum(row[f'var_{f}'] for f in FEATURE_4D)
    row['mean_duration_ms'] = float(grp['duration_ms'].mean())
    rows.append(row)

artist_stats = pd.DataFrame(rows).sort_values('total_variance')

print('\nTop 20 LOWEST-variance artists (potential ghosts):')
low_var = artist_stats.head(20)
print(low_var[['artist','track_count','total_variance','mean_duration_ms']].to_string(index=False))

In [ ]:
print('\nTop 20 HIGHEST-variance artists (likely organic):')
high_var = artist_stats.tail(20).sort_values('total_variance', ascending=False)
print(high_var[['artist','track_count','total_variance','mean_duration_ms']].to_string(index=False))

In [ ]:
# Variance distribution plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(artist_stats['total_variance'], bins=80, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(0.01, color='red', linestyle='--', linewidth=2, label='Ghost threshold (0.01)')
ax.axvline(artist_stats['total_variance'].median(), color='green', linestyle='--',
           linewidth=2, label=f'Median ({artist_stats["total_variance"].median():.3f})')
ax.set_xlabel('Total Variance (sum of 4 feature variances)', fontsize=12)
ax.set_ylabel('Number of Artists')
ax.set_title('Artist Catalog Variance Distribution\nLeft tail = most suspicious (uniform catalog)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'kaggle_variance_distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: kaggle_variance_distribution.png')

In [ ]:
# Save outputs
low_var_full = artist_stats.head(100)
high_var_full = artist_stats.tail(100).sort_values('total_variance', ascending=False)

low_var_full.to_csv(PROCESSED / 'low_variance_artists.csv', index=False)
high_var_full.to_csv(PROCESSED / 'high_variance_artists.csv', index=False)

print(f'Saved low_variance_artists.csv  ({len(low_var_full)} artists)')
print(f'Saved high_variance_artists.csv ({len(high_var_full)} artists)')

## Summary

In [ ]:
print('=== KAGGLE DATASET SUMMARY ===')
print(f'Total tracks:          {len(df):,}')
print(f'Unique track IDs:      {df["track_id"].nunique():,}')
print(f'Unique artists:        {df["artists"].nunique():,}')
print(f'Unique genres:         {df["track_genre"].nunique()}')
print(f'Artists with 10+ trks: {len(artists_10plus):,}')
print(f'Missing values:        {df.isnull().sum().sum()} (negligible)')
print()
print('Top 5 lowest-variance (ghost candidates):')
for _, r in artist_stats.head(5).iterrows():
    print(f'  {r.artist:<40} var={r.total_variance:.5f}  tracks={int(r.track_count)}')
print()
print('Top 5 highest-variance (organic controls):')
for _, r in artist_stats.tail(5).sort_values('total_variance', ascending=False).iterrows():
    print(f'  {r.artist:<40} var={r.total_variance:.5f}  tracks={int(r.track_count)}')